# Patchsorter distribution plot tile server
Patchsorter will have a tile server that produces raster tiles of the distribution.

The tile server will use the following URL pattern:
`/tiles/{z}/{x}/{y}`

Where:
- `z` is the zoom level (0-18)
- `x` is the tile column
- `y` is the tile row

Internally, tiles will be generated on demand. 

In [ ]:
def get_tile(z, x, y):
    # Placeholder function to generate a tile based on z, x, y
    # In a real implementation, this would generate a raster tile based on the distribution data
    return f"Tile at zoom {z}, column {x}, row {y}"

In [ ]:
import sys
import math
import psycopg2
from typing import Tuple, List, Optional, Dict, Any
from datetime import datetime

sys.path.insert(0, '..')
from utils import HierarchicalGridIndexZOrder, DB_PARAMS


In [ ]:
class PatchAggregationStore:
    """
    Queries patch aggregation data from per-level database tables.

    Each aggregation level maps to a dedicated table named
    ``patch_aggregation_level_{level}``. The ``grid_cell_id`` column stores
    Morton-encoded cell indices produced by ``HierarchicalGridIndexZOrder``.

    Table schema
    ------------
    grid_cell_id  INT        (PK)  Morton-encoded cell identifier
    bucket_date   TIMESTAMP  (PK)  Date the bucket was last updated
    pred_label    INT        (PK)  Predicted class label
    gt_label      INT        (PK)  Ground-truth class label
    count         INT              Number of patches in the bucket
    """

    TABLE_PREFIX = "patch_aggregation_level_"

    def __init__(
        self,
        cell_size: float = 1.0,
        db_params: Optional[Dict[str, Any]] = None,
    ) -> None:
        """
        Args:
            cell_size: Base cell size at level 0 (same units as patch coordinates).
            db_params: psycopg2 connection kwargs; falls back to ``DB_PARAMS``
                       from utils if not provided.
        """
        self.grid_index = HierarchicalGridIndexZOrder(cell_size=cell_size)
        self.db_params = db_params or DB_PARAMS

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def get_region(
        self,
        level: int,
        bbox: Tuple[float, float, float, float],
        bucket_date: Optional[datetime] = None,
    ) -> List[Dict[str, Any]]:
        """
        Return all aggregation buckets whose grid cells intersect *bbox*.

        Args:
            level:       Aggregation level; selects the table and cell resolution.
            bbox:        ``(min_x, min_y, max_x, max_y)`` in patch-coordinate space.
            bucket_date: Optional exact ``bucket_date`` filter.

        Returns:
            List of dicts, one per matching row::

                {
                    "grid_cell_id": int,
                    "bucket_date":  datetime,
                    "pred_label":   int,
                    "gt_label":     int,
                    "count":        int,
                    "center_x":     float,   # cell centre in patch-coordinate space
                    "center_y":     float,
                }
        """
        cell_ids = self._cells_in_bbox(level, bbox)
        if not cell_ids:
            return []

        table = self._table_name(level)
        rows = self._query(table, cell_ids, bucket_date)

        results = []
        for row in rows:
            grid_cell_id, bd, pred_label, gt_label, count = row
            cx, cy = self.grid_index.cell_to_point(grid_cell_id)
            results.append({
                "grid_cell_id": grid_cell_id,
                "bucket_date":  bd,
                "pred_label":   pred_label,
                "gt_label":     gt_label,
                "count":        count,
                "center_x":     cx,
                "center_y":     cy,
            })

        return results

    # ------------------------------------------------------------------
    # Private helpers
    # ------------------------------------------------------------------

    def _table_name(self, level: int) -> str:
        return f"{self.TABLE_PREFIX}{level}"

    def _cells_in_bbox(
        self, level: int, bbox: Tuple[float, float, float, float]
    ) -> List[int]:
        """
        Enumerate Morton-coded cell IDs that cover *bbox* at *level*.

        The bounding box is treated as a half-open interval
        ``[min_x, max_x) × [min_y, max_y)``.
        """
        min_x, min_y, max_x, max_y = bbox
        cell_size = self.grid_index.cell_size / (2 ** level)

        i_min = math.floor(min_x / cell_size)
        i_max = math.floor((max_x - 1e-10) / cell_size)
        j_min = math.floor(min_y / cell_size)
        j_max = math.floor((max_y - 1e-10) / cell_size)

        cell_ids = []
        for i in range(i_min, i_max + 1):
            for j in range(j_min, j_max + 1):
                morton = HierarchicalGridIndexZOrder._encode_morton(i, j)
                cell_ids.append((level << 58) | morton)

        return cell_ids

    def _query(
        self,
        table: str,
        cell_ids: List[int],
        bucket_date: Optional[datetime],
    ) -> List[tuple]:
        """Execute the SELECT against *table* and return raw rows."""
        sql = f"""
            SELECT grid_cell_id, bucket_date, pred_label, gt_label, count
            FROM   {table}
            WHERE  grid_cell_id = ANY(%s)
        """
        params: list = [cell_ids]

        if bucket_date is not None:
            sql += " AND bucket_date = %s"
            params.append(bucket_date)

        with psycopg2.connect(**self.db_params) as conn:
            with conn.cursor() as cur:
                cur.execute(sql, params)
                return cur.fetchall()
